# 🎗️ Project: SIIM-ISIC Melanoma Classification

**Dataset Link:** SIIM-ISIC Melanoma Classification Data

**📚 About the Dataset**

This dataset challenges you to identify Melanoma (a deadly skin cancer) in images of skin lesions.

   **Images:** The dataset contains thousands of dermatoscopic images (JPEGs and TFRecords).

   **Metadata:** Crucial patient information like sex, age_approx, and anatom_site (where the mole is located).

   **Class Imbalance:** Most images are benign (harmless). Only a tiny fraction are malignant (cancerous). This makes the problem hard!

# Step 1: Imports & Configuration

**IMG_SIZE = 512:** The code resizes all images to 512x512 pixels. This is a high resolution that allows the model to see fine texture details in the skin lesions.

**N_FOLDS = 5:** We are using 5-fold cross-validation. This means we train the model 5 times on different parts of the data to ensure it's robust.

In [ ]:
import numpy as np
import pandas as pd
import os
import re
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0, MobileNetV3Small
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder

IMG_SIZE = 512    
BATCH_SIZE = 16  
EPOCHS = 5       
SEED = 42
N_FOLDS = 5       
FOLD_TO_TRAIN = 0 

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("--- Step 1: Imports & Config Complete (Memory Safe Mode) ---")
print(f"TensorFlow Version: {tf.__version__}")

2025-11-20 04:13:05.912009: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763611986.136494      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763611986.202139      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

--- Step 1: Imports & Config Complete (Memory Safe Mode) ---
TensorFlow Version: 2.18.0


# 🏎️ Step 2: Hardware Acceleration Strategy

**Why this matters:** Deep learning on large images is slow.

**TPU (Tensor Processing Unit):** A super-fast chip made by Google specifically for AI.

**The Logic: The code checks "Am I on a TPU?".** If yes, it connects to it. If not, it falls back to the GPU (like the NVIDIA T4 that device are using). This makes your code portable.

In [ ]:
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    print('Running on TPU ', tpu.master())
except ValueError:
    tpu = None

if tpu:
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
else:
    strategy = tf.distribute.get_strategy()

AUTO = tf.data.AUTOTUNE
REPLICAS = strategy.num_replicas_in_sync
# We scale the batch size by the number of cores
BATCH_SIZE = BATCH_SIZE * REPLICAS
print(f"REPLICAS: {REPLICAS}, New Batch Size: {BATCH_SIZE}")

from kaggle_datasets import KaggleDatasets
COMP_PATH = 'siim-isic-melanoma-classification'
GCS_PATH = KaggleDatasets().get_gcs_path(COMP_PATH)

TRAIN_TFREC_PATHS = tf.io.gfile.glob(GCS_PATH + '/tfrecords/train*.tfrec')
TEST_TFREC_PATHS = tf.io.gfile.glob(GCS_PATH + '/tfrecords/test*.tfrec')
print(f"Found {len(TRAIN_TFREC_PATHS)} training TFRecord files.")
print(f"Found {len(TEST_TFREC_PATHS)} test TFRecord files.")

BASE_PATH = f'/kaggle/input/{COMP_PATH}'
df_train = pd.read_csv(os.path.join(BASE_PATH, 'train.csv'))
df_test = pd.read_csv(os.path.join(BASE_PATH, 'test.csv'))
df_sub = pd.read_csv(os.path.join(BASE_PATH, 'sample_submission.csv'))

print("--- Step 2: TPU Setup & Data Paths Complete ---")

REPLICAS: 1, New Batch Size: 16
Found 16 training TFRecord files.
Found 16 test TFRecord files.
--- Step 2: TPU Setup & Data Paths Complete ---


# 🛡️ Step 3: "Group" Splitting

**The Problem:** A single patient might have 10 different photos of moles. If you put 5 photos in the "Train" set and 5 in the "Test" set, the model will just memorize the patient's skin color instead of learning what cancer looks like. This is called Data Leakage.

**The Solution (GroupKFold):** It ensures that all photos of the same patient go into the SAME split. If Patient A is in the training set, none of their photos will appear in the validation set.

In [ ]:
print("Creating patient-level validation split...")
gkf = GroupKFold(n_splits=N_FOLDS)
df_train['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(
    gkf.split(X=df_train, y=df_train['target'], groups=df_train['patient_id'])
):
    df_train.loc[val_idx, 'fold'] = fold

fold = FOLD_TO_TRAIN
print(f"Using Fold {fold} for validation.")

train_image_names = df_train[df_train['fold'] != fold]['image_name'].values
val_image_names = df_train[df_train['fold'] == fold]['image_name'].values

def get_tfrec_paths(image_names, all_tfrec_paths):
    paths = []
    tfrec_indices = {int(path.split('-')[-1].split('.')[0]): path for path in all_tfrec_paths}
    
    image_indices = df_train[df_train['image_name'].isin(image_names)].index.tolist()
    
    for idx in image_indices:
        if idx in tfrec_indices:
            paths.append(tfrec_indices[idx])
            
    return list(set(paths)) 

# We can simplify this by just splitting the TFRec files by fold
train_files = [path for i, path in enumerate(TRAIN_TFREC_PATHS) if i % N_FOLDS != fold]
val_files = [path for i, path in enumerate(TRAIN_TFREC_PATHS) if i % N_FOLDS == fold]


print(f"Training file count: {len(train_files)}")
print(f"Validation file count: {len(val_files)}")
print("--- Step 3: Validation Split Complete ---")

Creating patient-level validation split...
Using Fold 0 for validation.
Training file count: 12
Validation file count: 4
--- Step 3: Validation Split Complete ---


# 🧹 Step 4: Metadata Cleaning

**Missing Data:** The dataset has holes (e.g., some patients didn't say their age).

## The Fix:

   **Categorical (Sex/Site):** Fill with "unknown".

   **Numerical (Age):** Fill with the average age of all patients. Deep learning models crash if you feed them "NaN" (Not a Number).

In [ ]:
# Metadata Preprocessing

print("Fitting metadata preprocessors...")
# Fill NaNs
df_train['sex'] = df_train['sex'].fillna('unknown')
df_train['anatom_site_general_challenge'] = df_train['anatom_site_general_challenge'].fillna('unknown')
df_train['age_approx'] = df_train['age_approx'].fillna(df_train['age_approx'].mean())
# Do the same for test data
df_test['sex'] = df_test['sex'].fillna('unknown')
df_test['anatom_site_general_challenge'] = df_test['anatom_site_general_challenge'].fillna('unknown')
df_test['age_approx'] = df_test['age_approx'].fillna(df_train['age_approx'].mean()) # Use train mean

# Fit OneHotEncoder
categorical_features = ['sex', 'anatom_site_general_challenge']
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(df_train[categorical_features])

# Fit StandardScaler
numerical_features = ['age_approx']
scaler = StandardScaler()
scaler.fit(df_train[numerical_features])

# Get the total number of tabular features
TAB_FEATURES_COUNT = ohe.categories_[0].shape[0] + ohe.categories_[1].shape[0] + len(numerical_features)
print(f"Total tabular features after encoding: {TAB_FEATURES_COUNT}")

AGE_MEAN = scaler.mean_[0]
AGE_STD = scaler.scale_[0]
SEX_CATS = ohe.categories_[0].tolist()
SITE_CATS = ohe.categories_[1].tolist()

print("--- Step 4: Metadata Preprocessing Complete ---")

Fitting metadata preprocessors...
Total tabular features after encoding: 11
--- Step 4: Metadata Preprocessing Complete ---


# 🏭 Step 5: The TFRecord Pipeline

This is the engine of your code. Instead of reading thousands of small JPEGs (which is slow), it reads **TFRecords** (large files containing many images packed together).

## 5.1 Parsing the Data

It tells TensorFlow exactly what is inside the binary files: a raw image string, a number for sex, a number for age, etc.

## 5.2 Augmentation (Making the model tougher)

**Why?** If you only show the model upright photos, it won't recognize a mole if the photo is taken upside down.

**Augmentation:** It randomly flips, rotates, and zooms into images during training. This creates "new" data for free and prevents overfitting.

In [ ]:
# Define the features in the TFRecord file
# We remove unused fields (patient_id, diagnosis) to reduce errors
TFREC_FEATURES = {
    'image': tf.io.FixedLenFeature([], tf.string),
    # We add default_value to handle missing data gracefully
    'sex': tf.io.FixedLenFeature([], tf.int64, default_value=0),
    'age_approx': tf.io.FixedLenFeature([], tf.float32, default_value=0.0),
    'anatom_site_general_challenge': tf.io.FixedLenFeature([], tf.int64, default_value=0),
    'target': tf.io.FixedLenFeature([], tf.int64, default_value=0)
}

# Test TFRecords have different features
TFREC_FEATURES_TEST = {
    'image': tf.io.FixedLenFeature([], tf.string),
    'sex': tf.io.FixedLenFeature([], tf.int64, default_value=0),
    'age_approx': tf.io.FixedLenFeature([], tf.float32, default_value=0.0),
    'anatom_site_general_challenge': tf.io.FixedLenFeature([], tf.int64, default_value=0)
}

def decode_image(image_bytes):
    """Decodes JPEG bytes, resizes, and normalizes."""
    image = tf.image.decode_jpeg(image_bytes, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32)
    image = image / 255.0  # Normalize to [0, 1]
    return image

def get_data_augmentation():
    """Returns a sequential model of augmentation layers."""
    return keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical", seed=SEED),
        layers.RandomRotation(0.2, seed=SEED),
        layers.RandomZoom(0.2, seed=SEED),
        layers.RandomContrast(0.2, seed=SEED),
        layers.RandomBrightness(0.2, seed=SEED),
    ], name="data_augmentation")

data_aug = get_data_augmentation()

def process_metadata(sex, age, site):
    """Applies OHE and scaling in a TensorFlow-native way."""
    
    # 1. One-hot encode 'sex'
    # The TFRecord stores 0, 1, 2. We match these to the categories.
    sex_ohe = tf.one_hot(tf.cast(sex, tf.int32), depth=len(SEX_CATS))
    
    # 2. One-hot encode 'anatom_site'
    site_ohe = tf.one_hot(tf.cast(site, tf.int32), depth=len(SITE_CATS))
    
    # 3. Scale 'age' using the pre-calculated mean and std
    age_scaled = (age - AGE_MEAN) / AGE_STD
    age_scaled = tf.expand_dims(age_scaled, axis=0) # Make it 1D
    
    # 4. Concatenate
    return tf.concat([sex_ohe, site_ohe, age_scaled], axis=0)

def read_tfrecord(example, is_training):
    """Reads, parses, and processes a single TFRecord example."""
    if is_training:
        features = tf.io.parse_single_example(example, TFREC_FEATURES)
        target = tf.cast(features['target'], tf.float32)
    else:
        features = tf.io.parse_single_example(example, TFREC_FEATURES_TEST)
        target = None # No target in test data

    image = decode_image(features['image'])
    
    # Get metadata features
    sex = features['sex']
    age = features['age_approx']
    site = features['anatom_site_general_challenge']
    
    # Process metadata
    metadata = process_metadata(sex, age, site)
    
    if is_training:
        return (image, metadata), target
    else:
        # For test data, we only need the inputs
        return (image, metadata), None

def get_dataset(filenames, is_training=True, augment=False):
    """Creates a complete, batched, prefetched dataset."""
    dataset = tf.data.TFRecordDataset(filenames, num_parallel_reads=AUTO)
    
    # Set options to ensure deterministic ordering during testing
    options = tf.data.Options()
    options.experimental_deterministic = not is_training
    dataset = dataset.with_options(options)
    
    # Parse and process
    dataset = dataset.map(lambda x: read_tfrecord(x, is_training), num_parallel_calls=AUTO)
    
    if is_training:
        # Apply augmentation ONLY to the image part
        if augment:
            dataset = dataset.map(lambda x, y: ((data_aug(x[0]), x[1]), y), num_parallel_calls=AUTO)
        
        # Shuffle
        dataset = dataset.shuffle(2048)
    
    # Batch and prefetch
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTO)
    return dataset

print("--- Step 5: Data Pipeline (ROBUST) Defined ---")

--- Step 5: Data Pipeline (ROBUST) Defined ---


# Step 6: Focal Loss

**The Issue:** In this dataset, 98% of images are healthy. A lazy model can just guess "Healthy" for everything and get 98% accuracy, but it will miss every cancer case.

**Focal Loss:** It acts like a strict teacher. It down-weights the easy examples (healthy skin) and heavily penalizes the model if it gets a hard example (cancer) wrong. It forces the model to focus on the difficult cases.

In [ ]:
# Define Focal Loss

def focal_loss(gamma=2.0, alpha=0.5):
    """
    Implementation of Focal Loss.
    gamma: focuses on hard examples.
    alpha: balances the importance of positive/negative examples.
    """
    def focal_loss_fixed(y_true, y_pred):
        # Ensure predictions are clipped to avoid log(0)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        
        # Calculate pt for positive class (target=1)
        pt_1 = tf.where(tf.equal(y_true, 1), y_pred, tf.ones_like(y_pred))
        # Calculate pt for negative class (target=0)
        pt_0 = tf.where(tf.equal(y_true, 0), y_pred, tf.zeros_like(y_pred))
        
        # Positive class loss
        loss_1 = -alpha * tf.pow(1.0 - pt_1, gamma) * tf.math.log(pt_1)
        # Negative class loss
        loss_0 = -(1 - alpha) * tf.pow(pt_0, gamma) * tf.math.log(1.0 - pt_0)
        
        # Sum the losses
        loss = loss_1 + loss_0
        return tf.reduce_mean(loss)
    return focal_loss_fixed

print("--- Step 6: Focal Loss Defined ---")

--- Step 6: Focal Loss Defined ---


# 🧠 Step 7: The Hybrid Model

This is a smart, modern architecture that uses both the image and the patient info.

## Branch A: The Eye (CNN)

Uses **MobileNetV3**, a lightweight and fast pre-trained model. It looks at the mole patterns.

## Branch B: The Context (Metadata)

It takes the patient's Age, Sex, and Mole Location.

**Why?** Melanoma is more common in older people and on certain body parts. This gives the model a "hint."

### Merging

It combines the image features and the metadata features into one final prediction.


In [ ]:
#Build the Model


with strategy.scope():
    def build_model(tab_features_count):
        # --- Image Branch (CNN) ---
        image_input = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image_input")
        
        base_model = MobileNetV3Small(
            weights='imagenet',
            include_top=False,
            input_tensor=image_input
        )
        base_model.trainable = True # We will fine-tune
        
        # Image Head
        x_img = layers.GlobalAveragePooling2D()(base_model.output)
        x_img = layers.Dense(128, activation='relu')(x_img)
        x_img = layers.Dropout(0.3)(x_img)
        
        # --- Tabular Branch (MLP) ---
        metadata_input = layers.Input(shape=(tab_features_count,), name="metadata_input")
        x_tab = layers.Dense(64, activation='relu')(metadata_input)
        x_tab = layers.Dropout(0.3)(x_tab)
        x_tab = layers.Dense(32, activation='relu')(x_tab)
        
        # --- Concatenation ---
        concatenated = layers.concatenate([x_img, x_tab], name="concatenation")
        
        # --- Final Classification Head ---
        x = layers.Dense(128, activation='relu')(concatenated)
        x = layers.Dropout(0.5)(x)
        final_output = layers.Dense(1, activation='sigmoid', name="output")(x)
        
        model = Model(inputs=[image_input, metadata_input], outputs=final_output)
        
        # Compile the model with Focal Loss and AUC metric
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-4),
            loss=focal_loss(),
            metrics=[
                tf.keras.metrics.AUC(name='auc'),
                'accuracy'
            ]
        )
        return model

    # Build the model
    model = build_model(TAB_FEATURES_COUNT)

print("--- Step 7: Model Built and Compiled ---")
# model.summary()

/usr/local/lib/python3.12/site-packages/keras/src/applications/mobilenet_v3.py:454: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
--- Step 7: Model Built and Compiled ---


# 🏋️ Step 8: Training

It trains on the **train_dataset** and evaluates on the **val_dataset**.

**Metric:** It watches val_auc (Area Under the Curve), which is better than accuracy for imbalanced data.

**Safety Net:** If the model stops improving for 5 epochs, it stops early to save time and reverts to the best version it found.

In [ ]:
# --- Create the datasets ---
print("Creating training and validation datasets...")
# We apply augmentation to the training set
train_dataset = get_dataset(train_files, is_training=True, augment=True)
# No augmentation on the validation set
val_dataset = get_dataset(val_files, is_training=True, augment=False) 

# Stop training if 'val_auc' stops improving
early_stopping = EarlyStopping(
    monitor='val_auc',
    patience=5,
    mode='max',
    restore_best_weights=True # Restore the best model weights
)
# Reduce learning rate if training plateaus
reduce_lr = ReduceLROnPlateau(
    monitor='val_auc',
    factor=0.2,
    patience=2,
    mode='max',
    min_lr=1e-6
)
# Save the best model automatically
model_checkpoint = ModelCheckpoint(
    filepath='best_model.keras', # This is the file you will download
    monitor='val_auc',
    mode='max',
    save_best_only=True
)

print(f"\n--- Starting Model Training on Fold {fold} ---")
history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=val_dataset,
    callbacks=[early_stopping, reduce_lr, model_checkpoint]
)

print("\n--- Model Training Finished ---")
print(f"Best model saved to 'best_model.keras'")

Creating training and validation datasets...

--- Starting Model Training on Fold 0 ---
Epoch 1/5
   1554/Unknown 1155s 732ms/step - accuracy: 0.9763 - auc: 0.5272 - loss: 0.0184

/usr/local/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1554/1554 ━━━━━━━━━━━━━━━━━━━━ 1230s 781ms/step - accuracy: 0.9811 - auc: 0.5381 - loss: 0.0164 - val_accuracy: 0.9828 - val_auc: 0.4904 - val_loss: 0.0211 - learning_rate: 1.0000e-04
Epoch 2/5
1554/1554 ━━━━━━━━━━━━━━━━━━━━ 1209s 777ms/step - accuracy: 0.9822 - auc: 0.5637 - loss: 0.0148 - val_accuracy: 0.9828 - val_auc: 0.6499 - val_loss: 0.0138 - learning_rate: 1.0000e-04
Epoch 3/5
1554/1554 ━━━━━━━━━━━━━━━━━━━━ 1217s 782ms/step - accuracy: 0.9822 - auc: 0.5914 - loss: 0.0143 - val_accuracy: 0.9828 - val_auc: 0.5721 - val_loss: 0.0131 - learning_rate: 1.0000e-04
Epoch 4/5
1554/1554 ━━━━━━━━━━━━━━━━━━━━ 1205s 774ms/step - accuracy: 0.9822 - auc: 0.6068 - loss: 0.0140 - val_accuracy: 0.9828 - val_auc: 0.5984 - val_loss: 0.0129 - learning_rate: 1.0000e-04
Epoch 5/5
1554/1554 ━━━━━━━━━━━━━━━━━━━━ 1213s 780ms/step - accuracy: 0.9822 - auc: 0.6219 - loss: 0.0137 - val_accuracy: 0.9828 - val_auc: 0.7595 - val_loss: 0.0119 - learning_rate: 2.0000e-05

--- Model Training Finished ---
Best mo

# 📝 Step 9: Submission

**Load Best Model:** It ensures we use the smartest version of the model saved during training.

**Predict:** It runs the model on the hidden test images (from the competition).

**Save:** It writes the probabilities (0 to 1) into a CSV file that you can upload to Kaggle for scoring.

In [ ]:
print("Creating test dataset for submission...")
# Create the test dataset (no training, no augmentation)

try:
    if not model:
        model = tf.keras.models.load_model("best_model.keras",custom_objects={'focal_loss_fixed': focal_loss})
except:
    model = tf.keras.models.load_model("best_model.keras",custom_objects={'focal_loss_fixed': focal_loss})

# test_files = [path for i, path in enumerate(TEST_TFREC_PATHS) if i % N_FOLDS == fold]
test_dataset = get_dataset(TEST_TFREC_PATHS, is_training=False, augment=False)

print("Generating predictions on test set...")
# model.predict() will automatically use all TPU cores
predictions = model.predict(test_dataset)

# Format for submission
submission_preds = predictions.flatten()
df_sub['target'] = submission_preds
df_sub.to_csv('submission.csv', index=False)

print("\n--- SUBMISSION.CSV FILE CREATED! ---")
print("You are finished!")
print("Go to the 'Data' tab on the right, find the 'Output' section,")
print("and you will see 'submission.csv' and 'best_model.keras'.")
print("You can download both and submit the .csv file.")

Creating test dataset for submission...
Generating predictions on test set...
687/687 ━━━━━━━━━━━━━━━━━━━━ 62s 90ms/step

--- SUBMISSION.CSV FILE CREATED! ---
You are finished!
Go to the 'Data' tab on the right, find the 'Output' section,
and you will see 'submission.csv' and 'best_model.keras'.
You can download both and submit the .csv file.
